# Pt. 1 Ingesting Data & Sending to Pub/Sub

In [ ]:
pip install google-cloud-pubsub

In [ ]:
pip install google-cloud-pubsub gtfs-realtime-bindings

In [ ]:
import os
import time
import requests
import json
from google.cloud import pubsub_v1
from google.transit import gtfs_realtime_pb2

# 1. Setup Authentication & Pub/Sub Client
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/big-data-admin-key.json"

# 2. Setup Pub/Sub Client (Notice we removed the os.environ line!)
project_id = "bigdataproject-493002"
topic_id = "transit-realtime-data"

publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(project_id, topic_id)

def stream_to_pubsub(api_key, interval=30):
    url = f"https://api.511.org/Transit/VehiclePositions?api_key={api_key}&agency=RG"
    print(f"Starting real-time stream to Pub/Sub (polling every {interval}s)...")

    try:
        while True:
            response = requests.get(url)

            if response.status_code == 200:
                feed = gtfs_realtime_pb2.FeedMessage()
                feed.ParseFromString(response.content)

                publish_count = 0

                for entity in feed.entity:
                    if entity.HasField('vehicle'):
                        v = entity.vehicle

                        # 2. Package the data into a dictionary
                        vehicle_data = {
                            "vehicle_id": v.vehicle.id,
                            "route": v.trip.route_id,
                            "lat": v.position.latitude,
                            "lon": v.position.longitude,
                            "timestamp": v.timestamp
                        }

                        # 3. Convert to JSON, then encode to bytes
                        message_json = json.dumps(vehicle_data)
                        message_bytes = message_json.encode("utf-8")

                        # 4. Publish to Google Cloud Pub/Sub
                        publisher.publish(topic_path, data=message_bytes)
                        publish_count += 1

                print(f"[{time.strftime('%X')}] Successfully published {publish_count} vehicle updates to Pub/Sub.")
            else:
                print(f"Failed to fetch data: {response.status_code}")

            time.sleep(interval)

    except KeyboardInterrupt:
        print("\nStreaming stopped by user.")

# --- Execution ---
MY_API_KEY = "1c6a0e84-c0a5-44f1-aa5d-a092294a48d4"  # Replace with your actual key
stream_to_pubsub(MY_API_KEY)

Starting real-time stream to Pub/Sub (polling every 30s)...
[01:25:59] Successfully published 1434 vehicle updates to Pub/Sub.
[01:26:30] Successfully published 1437 vehicle updates to Pub/Sub.

Streaming stopped by user.


In [ ]:
subscription_id = "transit-realtime-data-sub" # Ensure this exactly matches the name you gave it in the Console

# 2. Create the Subscriber Client
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

# 3. Define what to do when a message arrives
def callback(message):
    try:
        # Decode the raw bytes back into a Python dictionary
        data = json.loads(message.data.decode("utf-8"))

        # Print the data to the screen (Later, this is where you'd write to a database)
        print(f"✅ Received: Vehicle {data.get('vehicle_id')} on route {data.get('route')} at ({data.get('lat')}, {data.get('lon')})")

        # VERY IMPORTANT: Acknowledge the message!
        # This tells Google Cloud to remove it from the queue so you don't process it twice.
        message.ack()

    except Exception as e:
        print(f"Error processing message: {e}")
        # If there is a crash, tell Pub/Sub we failed so it tries sending it again later
        message.nack()

print(f"Listening for messages on {subscription_path}...\n")

# 4. Open the channel and start listening
streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)

# 5. Keep the notebook cell running
try:
    # This blocks the cell from finishing, allowing the background listener to run indefinitely
    streaming_pull_future.result()
except KeyboardInterrupt:
    # Gracefully stop if you press the "Stop" button in Jupyter
    streaming_pull_future.cancel()
    print("\nSubscriber stopped by user.")

Listening for messages on projects/bigdataproject-493002/subscriptions/transit-realtime-data-sub...



PermissionDenied: 403 User not authorized to perform this action. [reason: "IAM_PERMISSION_DENIED"
domain: "iam.googleapis.com"
metadata {
  key: "permission"
  value: "pubsub.subscriptions.consume"
}
]

In [ ]:
# Part 3 -- Using H2o.ai to analyze foot traffic
import pandas as pd

# 1. Load your new BART dataset
df = pd.read_excel("combined_ridership_2025_full.xlsx")

# 2. Filter out "Total Trips" so we are only predicting daily averages
df = df[df['DayType'] != 'Total Trips'].copy()

# 3. Drop rows where Ridership is missing (there are about ~495 blank rows)
df = df.dropna(subset=['Ridership'])

# 4. Extract the Month from the 'Period' column (e.g., 202501 -> 1, 202512 -> 12)
df['Month'] = df['Period'].astype(str).str[-2:].astype(int)

# 5. Save the cleaned dataset for H2O
df.to_csv("bart_cleaned_for_h2o.csv", index=False)

print(f"Cleaned dataset ready with {len(df)} rows!")
print(df.head())

In [ ]:
pip install h2o

In [ ]:
import h2o
from h2o.automl import H2OAutoML

# 1. Initialize the H2O cluster
h2o.init()

# 2. Load the newly cleaned BART dataset
hf = h2o.import_file("bart_cleaned_for_h2o.csv")

# Ensure our text/category columns are treated as categories (factors) by H2O
hf["Origin"] = hf["Origin"].asfactor()
hf["Destination"] = hf["Destination"].asfactor()
hf["DayType"] = hf["DayType"].asfactor()
hf["Month"] = hf["Month"].asfactor() # Month is a category (Dec behaves differently than Jan)

# 3. Define what we are predicting (Response) and what we use to predict (Predictors)
response = "Ridership"
predictors = ["Origin", "Destination", "DayType", "Month"]

print(f"Predicting: {response}")
print(f"Using features: {predictors}")

# 4. Train/Test Split (80% training, 20% testing)
train, test = hf.split_frame(ratios=[0.8], seed=42)

# 5. Run AutoML
# We are predicting a continuous number, so we use RMSE to score the models
aml = H2OAutoML(
    max_runtime_secs=300, # Running for 5 minutes. You can increase to 600 if you want an even better model!
    seed=42,
    sort_metric="RMSE"
)

print("Training models... Let it run!")
aml.train(x=predictors, y=response, training_frame=train)

# 6. View the Best Models
print("\n--- AutoML Leaderboard ---")
print(aml.leaderboard.head(rows=10))

# 7. Evaluate the #1 Best Model
best_model = aml.leader
perf = best_model.model_performance(test)
print(f"\n--- Best Model Performance ({best_model.model_id}) ---")
print(perf)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import h2o

# 1. Extract the best single model (We still need this to get predictions!)
model_ids = aml.leaderboard['model_id'].as_data_frame()['model_id']
best_single_model_id = [m for m in model_ids if 'Ensemble' not in m][0]
best_interpretable_model = h2o.get_model(best_single_model_id)

print(f"Using {best_single_model_id} for visual interpretation...\n")

# 2. Variable Importance Plot (This native H2O function still works perfectly)
%matplotlib inline
best_interpretable_model.varimp_plot()

# ==========================================
# 3. GENERATING PREDICTIONS ON ORIGINAL DATA
# ==========================================
print("Generating predictions on the actual, unmodified test data...")

# Ask the model to predict ridership for the test set
preds = best_interpretable_model.predict(test).as_data_frame()

# Convert the original H2O test set back to a standard Pandas DataFrame
results_df = test.as_data_frame()

# Add the model's predictions as a new column right next to the original data
results_df['Predicted_Ridership'] = preds['predict']


# ==========================================
# 4. BUSINESS IMPACT: DAY TYPE
# ==========================================
# Group by the ORIGINAL DayType and average the predictions
day_type_impact = results_df.groupby('DayType')['Predicted_Ridership'].mean().reindex(
    ['Average Weekday', 'Average Saturday', 'Average Sunday']
)

plt.figure(figsize=(8, 5))
day_type_impact.plot(kind='bar', color=['#4CAF50', '#FFC107', '#F44336'])
plt.title('Average Predicted Ridership by Day Type', fontsize=14, fontweight='bold')
plt.ylabel('Average Ridership per Station', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# ==========================================
# 5. BUSINESS IMPACT: SEASONALITY (MONTH)
# ==========================================
# Group by the ORIGINAL Month and average the predictions
month_impact = results_df.groupby('Month')['Predicted_Ridership'].mean().sort_index()

plt.figure(figsize=(10, 5))
month_impact.plot(kind='line', marker='o', color='#2196F3', linewidth=3, markersize=8)
plt.title('Average Predicted Ridership by Month', fontsize=14, fontweight='bold')
plt.xlabel('Month (1 = Jan, 12 = Dec)', fontsize=12)
plt.ylabel('Average Ridership per Station', fontsize=12)
plt.xticks(month_impact.index)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
pip install folium

In [ ]:

'''
%pip install folium scikit-learn

import folium
from folium.plugins import MarkerCluster
import pandas as pd
import json
import time
from IPython.display import display, clear_output
from google.cloud import pubsub_v1

# For clustering to identify high-traffic areas
from sklearn.cluster import DBSCAN
import numpy as np

# Pub/Sub setup
# These values are taken from previous cells in the notebook.
project_id = "bigdataproject-493002"
topic_id = "transit-realtime-data" # This is for context, not directly used by subscriber
subscription_id = "transit-realtime-data-sub"

subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

# Global dictionary to store the latest vehicle positions
# Key: vehicle_id, Value: {'lat': ..., 'lon': ..., 'route': ..., 'timestamp': ...}
vehicle_positions = {}
last_message_time = time.time() # To track when the last message was received

# Define the callback function for Pub/Sub messages
def callback(message):
    global vehicle_positions
    global last_message_time
    try:
        data = json.loads(message.data.decode("utf-8"))
        vehicle_id = data.get('vehicle_id')
        if vehicle_id:
            vehicle_positions[vehicle_id] = {
                'lat': data.get('lat'),
                'lon': data.get('lon'),
                'route': data.get('route'),
                'timestamp': data.get('timestamp')
            }
        message.ack()
        last_message_time = time.time()
    except Exception as e:
        print(f"Error processing message: {e}")
        message.nack()

# Function to draw the map
def draw_live_map(current_vehicle_data, traffic_density_points=None):
    if not current_vehicle_data:
        # If no data, return a default centered map (e.g., San Francisco Bay Area)
        return folium.Map(location=[37.7749, -122.4194], zoom_start=12)

    # Convert dictionary to DataFrame for easier processing
    df_live_traffic = pd.DataFrame.from_dict(current_vehicle_data, orient='index')

    # Calculate map center based on current vehicle positions
    map_center = [df_live_traffic['lat'].mean(), df_live_traffic['lon'].mean()]
    live_map = folium.Map(location=map_center, zoom_start=12)

    # Add markers for each vehicle, using MarkerCluster for better performance with many points
    marker_cluster = MarkerCluster().add_to(live_map)
    for index, row in df_live_traffic.iterrows():
        folium.Marker(
            location=[row['lat'], row['lon']],
            popup=f"Vehicle ID: {index}<br>Route: {row['route']}<br>Lat: {row['lat']}<br>Lon: {row['lon']}<br>Timestamp: {pd.to_datetime(row['timestamp'], unit='s')}",
            tooltip=f"Vehicle ID: {index}"
        ).add_to(marker_cluster)

    # Add high-traffic areas as circles
    if traffic_density_points:
        for cluster_lat, cluster_lon, num_vehicles in traffic_density_points:
            folium.CircleMarker(
                location=[cluster_lat, cluster_lon],
                radius=min(20, num_vehicles * 2), # Radius proportional to number of vehicles, capped at 20
                color='red',
                fill=True,
                fill_color='red',
                fill_opacity=0.5,
                tooltip=f"{num_vehicles} vehicles in this area"
            ).add_to(live_map)

    return live_map

# Main loop to subscribe and update map
def live_map_updater(update_interval=30):
    global vehicle_positions
    global last_message_time

    # Start the subscriber in a non-blocking way
    streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)
    print(f"Listening for messages on {subscription_path} and updating map every {update_interval}s...\n")

    try:
        while True:
            current_time = time.time()
            # If no messages received for a prolonged period, clear vehicle positions
            if current_time - last_message_time > 60:
                vehicle_positions.clear()
                print(f"[{time.strftime('%X')}] No messages received for 60 seconds. Clearing vehicle positions.")

            # Get current vehicle positions from the global dictionary
            current_vehicles_list = list(vehicle_positions.values())
            traffic_density_points = []

            if current_vehicles_list:
                # Prepare data for DBSCAN clustering
                coords = np.array([[v['lat'], v['lon']] for v in current_vehicles_list])

                # Apply DBSCAN to find high-density clusters
                # eps: Maximum distance for points to be considered neighbors (in degrees)
                # min_samples: Minimum number of points in a cluster
                # These parameters may need tuning based on the specific geographical area and vehicle density.
                db = DBSCAN(eps=0.005, min_samples=3).fit(coords)
                labels = db.labels_

                unique_labels = set(labels)
                for k in unique_labels:
                    if k == -1: # -1 indicates noise points (vehicles not part of any cluster)
                        continue
                    class_member_mask = (labels == k)
                    cluster_coords = coords[class_member_mask]
                    if len(cluster_coords) >= 3: # Only consider clusters with 3 or more vehicles for highlighting
                        # Calculate centroid of the cluster for the circle's center
                        cluster_center_lat, cluster_center_lon = cluster_coords.mean(axis=0)
                        traffic_density_points.append([cluster_center_lat, cluster_center_lon, len(cluster_coords)])

            # Clear previous output and display the new map
            clear_output(wait=True)
            live_map = draw_live_map(vehicle_positions, traffic_density_points)
            display(live_map)
            print(f"[{time.strftime('%X')}] Map updated with {len(vehicle_positions)} vehicles. Found {len(traffic_density_points)} high traffic areas.")

            time.sleep(update_interval)

    except KeyboardInterrupt:
        print("\nMap updater stopped by user.")
        streaming_pull_future.cancel() # Stop the subscription
        streaming_pull_future.result(timeout=5) # Wait for the subscriber to finish any pending messages, with a timeout
        subscriber.close() # Close the subscriber client

# --- Execution ---
print("Starting real-time foot traffic map visualization.")
print("Please ensure the Pub/Sub data publisher (e.g., from cell 'vlpEUE3nmk5V') is running in a separate tab or process for data to appear.")
live_map_updater(update_interval=30)

'''

In [ ]:
# LLM integration

